# Causal MAS Distillation - v3 run book

Full rebuild. **Nothing in this notebook reads the old caches, the old
`traces.jsonl`, or the old checkpoints.** Everything is written under a fresh
`v3/` directory on Drive so the old artefacts stay untouched but unused.

### What changed since the last run

| | v2 (failed) | v3 (this notebook) |
|---|---|---|
| critic model | deepseek-v3.2 (same as solver) | **gpt-oss-120b** |
| protocol | stateless self-refinement | debate with a shared transcript |
| keys | one, rate limited | round robin over several |
| trainer | `04_train.py` (silently trained on 5 tokens) | `04b_train_ab.py` with a hard guard |
| arms | A, B | A, B, **C**, plus the untrained base |
| trace pool | 134 headroom problems | all 785 probed problems |
| eval set | 32 problems, CI +-9 pts | ~140+ problems |

### Budget

| step | requests | wall clock | cost |
|---|---|---|---|
| 0 provider audit | ~10 | 2 min | ~0 |
| 1 probe k=32 | ~3,200 | 40-70 min | ~$4 |
| 2 traces, 785 x 3 seeds | ~14,000 | 1.5-2.5 h | ~$18 |
| 3 validation | 0 | 2 min | free |
| 4-5 build datasets | 0 | 5 min | free |
| 6 dry run | 0 | 1 min | free |
| 7 train 4 configs x 3 seeds | 0 | 3-5 h GPU | free |
| 8 eval | 0 | 40 min GPU | free |

Steps 1-2 run on CPU runtime. Switch to a **T4 GPU** runtime only at step 7.

## Three rules, learned the expensive way

1. **Never put an API key in a cell.** Colab saves cell output into the
   `.ipynb`, and this notebook goes in the public repo. Keys come from
   `google.colab.userdata` and are passed as environment variables only.
2. **Fresh cache file, not "no cache".** The cache is not an optimisation, it
   is the crash-resume mechanism, and for the probe it literally *is* the
   Arm A dataset - `10_build_arm_a.py` reads Arm A back out of the probe
   cache. Point every step at a new file under `v3/` and keep those files.
3. **Never skip step 3 or step 6.** Both are free and both catch the exact
   failure modes that already cost a day: bad traces, and a trainer that
   silently trains on nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
RUN  = 'v3'
BASE = '/content/drive/MyDrive/cmd'
V3   = f'{BASE}/{RUN}'
CKPT = f'{V3}/ckpt'
for d in (V3, CKPT):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
os.environ['V3'] = V3
os.environ['CKPT'] = CKPT
print('run dir :', V3)
print('existing:', sorted(os.listdir(V3)) or '(empty, good)')

In [ ]:
%cd /content
![ -d causal-mas-distill ] || git clone https://github.com/Arshia-HZ/causal-mas-distill.git
%cd /content/causal-mas-distill
!git pull --ff-only || true
!pip -q install -r requirements-api.txt

# sanity: the v3 critical path must all be present
import pathlib
need = ['scripts/00a_probe_difficulty.py','scripts/00f_check_traces.py',
        'scripts/00g_diagnose_signal.py','scripts/00j_provider_audit.py',
        'scripts/01b_generate_traces.py','scripts/10_build_arm_a.py',
        'scripts/11_build_abc_datasets.py','scripts/04b_train_ab.py',
        'scripts/12_eval_abc.py','src/backends/multikey.py',
        'src/debate/harness_debate.py']
missing = [f for f in need if not pathlib.Path(f).exists()]
print('MISSING:', missing) if missing else print('all v3 scripts present')
assert not missing, 'push the missing files before continuing'

### Credentials

In Colab: key icon in the left sidebar -> **Add new secret** -> name it
`GC_API_KEYS`, value is every key separated by commas and no spaces:

```
sk-aaaa,sk-bbbb,sk-cccc
```

Toggle *Notebook access* on. The cell below reads it and never prints it.

In [ ]:
from google.colab import userdata
import os

os.environ['GC_API_KEYS'] = userdata.get('GC_API_KEYS')
os.environ['API_URL']     = 'https://api.generalcompute.com/v1'

# roles. solver and verifier stay on the model the probe was run with,
# otherwise probed_all.json and the Arm A pool stop being valid.
os.environ['MODEL_SOLVER']   = 'deepseek-v3.2'
os.environ['MODEL_CRITIC']   = 'gpt-oss-120b'
os.environ['MODEL_VERIFIER'] = 'deepseek-v3.2'

# the probe and 10_build_arm_a.py MUST agree on this number, because it is
# part of the cache key. Change it in one place only.
os.environ['PROBE_MT'] = '1024'
os.environ['TRACE_MT'] = '1024'

n = len([k for k in os.environ['GC_API_KEYS'].split(',') if k.strip()])
print(f'{n} API key(s) loaded')
assert n >= 1

---
## Step 0 - what does the provider actually allow? (2 min, ~free)

The dashboard says deepseek-v3.2 has a 32k context. The runtime error we
actually hit said `this model only supports 8192`. One of those is wrong, and
the answer decides `--max-tokens` and the verifier transcript budget. This
sends one deliberately oversized request per model and reads the provider's
own error message.

It also checks whether `max_tokens` is honoured at all - 47% of solver
messages in the last run blew past the cap we set, which was never explained.

In [ ]:
!python scripts/00j_provider_audit.py --mode limits \
  --models $MODEL_SOLVER $MODEL_CRITIC \
  --out results/provider_limits.json

**Read the output before continuing.** If the real context is 8192, drop
`TRACE_MT` to 768 and set `--max-rounds 2` in step 2, otherwise the verifier
will start failing on long transcripts the way `math_0498` did.

---
## Step 1 - probe the dataset (40-70 min, ~$4)

32 samples per problem on the solver model. Two outputs:

- `probed_all.json` - the pass rate of every problem, which defines the
  difficulty bands and is used by every later script for stratification.
- `v3/cache_probe.jsonl` - **this is Arm A.** 785 x 32 sampled solutions.
  Step 4 reads correct ones straight back out of it, for free.

In [ ]:
!python scripts/00a_probe_difficulty.py \
  --input data/math_problems.json \
  --output data/gate_problems_k32.json \
  --probed-out data/probed_all.json \
  --api-url $API_URL --model $MODEL_SOLVER \
  --k 32 --keep-min 0.0 --keep-max 1.0 --max-problems 100000 \
  --max-tokens $PROBE_MT \
  --cache-path $V3/cache_probe.jsonl

In [ ]:
# back up the two files that are expensive to recreate
!cp data/probed_all.json $V3/probed_all.json
!ls -la $V3

---
## Step 2 - generate the debate traces (1.5-2.5 h, ~$18)

This is the run that matters. Three things are different from last time.

**All 785 problems, not just the 134 "headroom" ones.** Last time traces
existed for only 134 problems and rejection sampling covered 763, so after
intersecting the two arms only 123 problems survived and Arm B ended up
training on 22 of them. Coverage, not tokens, was the binding constraint.

**The critic is a different model.** One model grading its own work produced a
dispute rate of 8% and caught 10% of genuinely wrong solutions. That is
self-refinement, and self-refinement on math is a known negative result
(Huang et al., ICLR 2024). A different model does not share the solver's blind
spots. This is also what makes the word "multi-agent" true in the thesis.

**The critic sees the whole transcript**, with a role system prompt, and every
generation carries a unique cache nonce so two seeds cannot collapse onto one
cached continuation.

`--resume` makes this safe against a Colab disconnect: rerun the same cell and
it skips the problems already written.

In [ ]:
!python scripts/01b_generate_traces.py \
  --problems data/probed_all.json \
  --output data/traces_v3.jsonl \
  --api-url $API_URL --api-keys-env GC_API_KEYS \
  --solver-model $MODEL_SOLVER \
  --critic-model $MODEL_CRITIC \
  --verifier-model $MODEL_VERIFIER \
  --critic-persona adversarial \
  --max-rounds 3 --n-solutions 3 --max-tokens $TRACE_MT \
  --concurrency-per-key 8 --problem-concurrency 16 \
  --cache-path $V3/cache_traces.jsonl \
  --resume

In [ ]:
!cp data/traces_v3.jsonl $V3/traces_v3.jsonl
!du -h $V3/traces_v3.jsonl

---
## Step 3 - validate before spending anything else (free, 2 min)

**Do not skip this.** These two scripts read files only.

In [ ]:
!python scripts/00f_check_traces.py \
  --traces data/traces_v3.jsonl \
  --targets data/probed_all.json \
  --max-tokens $TRACE_MT

!python scripts/00g_diagnose_signal.py \
  --traces data/traces_v3.jsonl \
  --probed data/probed_all.json \
  --max-tokens $TRACE_MT

### Gate. Compare against the v2 numbers.

| metric | v2 | v3 must reach | if it does not |
|---|---|---|---|
| critic dispute rate | 0.080 | **>= 0.25** | the critic swap failed; try `gemma-4-31B-it`, and reread the critic prompt |
| critic recall on wrong answers | 0.100 | **>= 0.30** | same |
| identical round-1 solvers | 0.0% | stay 0.0% | the seed fix regressed |
| debate accuracy - round-1 accuracy | +0.036 | **> 0** | the debate is destroying answers, not fixing them |
| problems with >=1 correct trace | 123 | **>= 600** | coverage is still the bottleneck |

The last row is the one that decides whether the A/B comparison is even
powered. Everything downstream intersects with it.

If the dispute rate is still under 0.15, **stop and tell me** rather than
spending GPU hours on it.

---
## Step 4 - Arm A, free, from the probe cache (5 min, $0)

Arm A is the honest baseline: ordinary rejection-sampling fine-tuning (STaR /
RFT). Take the correct solutions the solver already produced during the probe.
No debate, no critic. If Arm B cannot beat this, the debate is not earning its
cost as a data engine, and that is the whole hypothesis.

`--max-tokens` here must equal `PROBE_MT` from step 1 or every cache lookup
misses.

In [ ]:
!python scripts/10_build_arm_a.py \
  --cache-path $V3/cache_probe.jsonl \
  --problems data/probed_all.json \
  --model $MODEL_SOLVER --n-probe 32 --max-tokens $PROBE_MT \
  --out data/arm_a_pool.jsonl

Recovery rate must be near 100%. If it is 0%, the cache key does not match -
almost always because `--max-tokens` or `--model` differs from step 1, or
because the solve prompt changed. The prompt used for the probe is frozen as
`LEGACY_SOLVE_PROMPT` inside `10_build_arm_a.py`; do not "clean it up" into an
import.

---
## Step 5 - build the four training sets (free)

| arm | what the student is trained to produce | tests |
|---|---|---|
| base | nothing, untrained Qwen2.5-1.5B | is the pipeline even working |
| A | one correct solution from the solver | rejection-sampling baseline |
| B | the full debate transcript | is the *process* teachable |
| C | only the debate's final solution | does debating produce a *better product* |

B minus C is the interesting contrast. A win for B over C says the critique
text itself carries trainable signal. A win for C over A with no win for B
says debate improves the answer but the transcript is just noise to a 1.5B
student. Either is a real finding; the two-arm design could not tell them
apart.

The script enforces the traps that broke earlier runs: identical problem
coverage across arms, a matched completion-token budget, a real tokenizer
rather than `len//4`, and a held-out split by hash of the problem id.

In [ ]:
!python scripts/11_build_abc_datasets.py \
  --arm-a-pool data/arm_a_pool.jsonl \
  --traces data/traces_v3.jsonl \
  --probed data/probed_all.json \
  --clip-critic-words 200 \
  --max-rounds-render 2 \
  --max-completion-tokens 3584 \
  --eval-extra 200 \
  --budget-tokens 2000000 \
  --outdir data

!cp data/sft_arm_*_eqp.jsonl data/eval_problems.json $V3/ 2>/dev/null; ls $V3

The `_eqp` files are the primary variant: **equal problems per arm**, one
example per problem. Use those. The `_tok` files are the equal-token secondary
variant, reported as a robustness check.

Check the printed eval-set size. Under 100 problems and the confidence
interval will be wider than any effect worth reporting.

---
## Step 6 - dry run the trainer (free, 1 min) - MANDATORY

This is the step that did not exist on 13 Aug, which is why both arms trained
on the 5-token string `"Answer: "` and produced `loss 2e-07` in 19 seconds.
`--dry-run` tokenizes the dataset, prints the actual supervised span, and
**exits non-zero if fewer than 80% of examples survive**. It never touches the
GPU.

In [ ]:
import subprocess, sys
ok = True
for arm in ['a', 'b', 'c']:
    print('=' * 60, '\nDRY RUN arm', arm, flush=True)
    r = subprocess.run([sys.executable, 'scripts/04b_train_ab.py',
                        '--dataset', f'data/sft_arm_{arm}_eqp.jsonl',
                        '--dry-run'])
    ok &= (r.returncode == 0)
print('\nALL ARMS OK' if ok else '\nSTOP. Fix the dataset before training.')
assert ok

---
## Step 7 - train (3-5 h on a T4)

**Switch the runtime to GPU now** (Runtime -> Change runtime type -> T4), then
rerun the mount cell and the credentials cell, then this one.

Three seeds per arm. One seed is not a result: the seed-to-seed spread on a
1.5B LoRA is comparable to the effect we are trying to measure.

In [ ]:
!pip -q install -r requirements-train.txt
import torch
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability())
# T4 is sm_75 and has no bf16. 04b_train_ab.py detects this; the old yaml did not.

In [ ]:
import os, pathlib, subprocess, sys, time

CKPT = os.environ['CKPT']
t0 = time.time()
for arm in ['a', 'b', 'c']:
    for seed in [0, 1, 2]:
        out = f'{CKPT}/arm_{arm}/seed{seed}'
        if pathlib.Path(f'{out}/final/adapter_model.safetensors').exists():
            print(f'skip arm_{arm} seed{seed}')
            continue
        print('=' * 60, f'\ntraining arm_{arm} seed{seed}  [{(time.time()-t0)/60:.0f} min in]', flush=True)
        subprocess.run([sys.executable, 'scripts/04b_train_ab.py',
                        '--dataset', f'data/sft_arm_{arm}_eqp.jsonl',
                        '--output-dir', out,
                        '--max-seq-length', '4096',
                        '--epochs', '3',
                        '--seed', str(seed)], check=True)
print(f'done in {(time.time()-t0)/60:.0f} min')

Sanity check while it runs: a real run reports thousands of `num_tokens`, a
loss that decreases over several hundred steps, and a runtime in the tens of
minutes. If any arm finishes in under two minutes, stop - step 6 was bypassed.

---
## Step 8 - evaluate (40 min on the T4)

Greedy decoding, held-out problems only, three seeds per arm averaged per
problem, paired bootstrap over problems.

In [ ]:
CK = os.environ['CKPT']
seeds = lambda a: ' '.join(f'{CK}/arm_{a}/seed{s}/final' for s in (0,1,2))
cmd = f"""python scripts/12_eval_abc.py \
  --eval data/eval_problems.json \
  --arm-base base \
  --arm-a {seeds('a')} \
  --arm-b {seeds('b')} \
  --arm-c {seeds('c')} \
  --probed data/probed_all.json \
  --out results/abc_eval.json"""
print(cmd)
get_ipython().system(cmd)

In [ ]:
!cp results/abc_eval.json $V3/abc_eval.json
import json; print(json.dumps(json.load(open('results/abc_eval.json')), indent=2)[:3000])

### How to read it

**Read the BASE row first.** If A, B and C are all at or below the untrained
base model, the training pipeline is broken and nothing else in the table
means anything.

Then, in order:

1. `A - BASE` - does supervised fine-tuning on teacher solutions help at all?
   If not, nothing downstream is interpretable.
2. `B - A` - **the thesis question.** Is a debate transcript better training
   data than a plain correct solution at matched budget?
3. `C - A` - does the debate at least produce better *answers*?
4. `B - C` - is the critique text itself worth anything to the student?

A confidence interval that spans zero is a null, not a small effect. With
~140 eval problems the interval is about +-4 points, so anything under that is
unmeasurable at this scale and should be reported as such rather than spun.

---
## Phase 3 - only if B or C wins

The original thesis idea - counterfactual message utility as a data-selection
signal - is a *refinement* of "debate transcripts are good training data". It
only has an audience if that premise survives step 8. Do not start it before
then.

When the time comes the order is: `placebo_check` first (it must return
exactly 0.0), then `trace_utilities_total_crn --k 32`, and the biased `total`
estimand alongside it as evidence that common random numbers mattered.

If B and C both lose, the paper is not dead. "Debate is a worse data engine
than rejection sampling at matched budget, and here is the coverage analysis
showing why" is a defensible negative result, and the coverage-gate numbers
from `00i` already support it.